In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("ab_data.csv")

In [4]:
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [5]:
df.shape

(294478, 5)

In [6]:

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   user_id       294478 non-null  int64
 1   timestamp     294478 non-null  str  
 2   group         294478 non-null  str  
 3   landing_page  294478 non-null  str  
 4   converted     294478 non-null  int64
dtypes: int64(2), str(3)
memory usage: 11.2 MB


In [7]:

df.isnull().sum()


user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [8]:
df["user_id"].duplicated().sum()

np.int64(3894)

In [9]:
df["user_id"].nunique()

290584

In [10]:

df["group"].value_counts()


group
treatment    147276
control      147202
Name: count, dtype: int64

In [11]:

df["landing_page"].value_counts()


landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64

In [12]:

pd.crosstab(df["group"], df["landing_page"])


landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965


In [13]:

df_clean = df.copy()

print("Original rows:", len(df))
print("Working copy rows:", len(df_clean))


Original rows: 294478
Working copy rows: 294478


In [14]:

before = len(df_clean)

df_clean = df_clean[
    ((df_clean["group"] == "control") & (df_clean["landing_page"] == "old_page")) |
    ((df_clean["group"] == "treatment") & (df_clean["landing_page"] == "new_page"))
].copy()

after = len(df_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)


Rows before: 294478
Rows after: 290585
Rows removed: 3893


In [15]:

df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])

print(df_clean["timestamp"].dtype)


datetime64[us]


In [16]:

before = len(df_clean)

df_clean = df_clean.sort_values("timestamp")

df_clean = df_clean.drop_duplicates(
    subset="user_id",
    keep="first"
).copy()

after = len(df_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicate rows removed:", before - after)


Rows before: 290585
Rows after: 290584
Duplicate rows removed: 1


In [17]:

print("Total rows:", len(df_clean))

print("Duplicate users:",
      df_clean["user_id"].duplicated().sum())

print("Group and page matching:")
print(pd.crosstab(df_clean["group"], df_clean["landing_page"]))


Total rows: 290584
Duplicate users: 0
Group and page matching:
landing_page  new_page  old_page
group                           
control              0    145274
treatment       145310         0


In [18]:

df_clean.to_csv("ab_data_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")


Cleaned dataset saved successfully!



## Data Cleaning Log

Original dataset: 294,478 rows

### Cleaning Steps
1. Created a copy of the original dataset.
2. Removed 3,893 mismatched rows.
3. Converted timestamp to datetime.
4. Sorted records by timestamp.
5. Removed 1 duplicate row, keeping the earliest recorded row.

### Final Results
- Cleaned dataset: 290,584 rows
- Duplicate users: 0
- Group/page mismatches: 0
- Missing values: 0

### Limitations
- Removing mismatched rows may introduce selection bias.
- Earliest recorded timestamp is treated as first exposure.
- This is a public sample dataset, not real company data.


In [19]:

group_counts = df_clean["group"].value_counts()

print(group_counts)


group
treatment    145310
control      145274
Name: count, dtype: int64


In [20]:

group_percentages = (group_counts / group_counts.sum()) * 100

print(group_percentages)


group
treatment    50.006194
control      49.993806
Name: count, dtype: float64


In [21]:

from scipy.stats import chisquare

original_counts = df["group"].value_counts()

observed = original_counts.reindex(
    ["control", "treatment"]
).values

expected = [observed.sum() / 2] * 2

chi2, p_value = chisquare(
    f_obs=observed,
    f_exp=expected
)

print("Chi-square statistic:", chi2)
print("SRM p-value:", p_value)


Chi-square statistic: 0.018595616650479832
SRM p-value: 0.8915321515063552


In [22]:

conversion_rates = df_clean.groupby("group")["converted"].mean() * 100

print(conversion_rates)


group
control      12.038630
treatment    11.880807
Name: converted, dtype: float64


In [23]:

control = conversion_rates["control"]
treatment = conversion_rates["treatment"]

absolute_lift = treatment - control

relative_lift = (absolute_lift / control) * 100

print("Absolute Lift:", round(absolute_lift, 4), "percentage points")
print("Relative Lift:", round(relative_lift, 4), "%")


Absolute Lift: -0.1578 percentage points
Relative Lift: -1.311 %


In [24]:

from statsmodels.stats.proportion import proportions_ztest

summary = df_clean.groupby("group")["converted"].agg(["sum", "count"])

conversions = summary.loc[["control", "treatment"], "sum"]
users = summary.loc[["control", "treatment"], "count"]

z_stat, p_value = proportions_ztest(
    count=conversions,
    nobs=users,
    alternative="two-sided"
)

print("Z-statistic:", z_stat)
print("P-value:", p_value)


Z-statistic: 1.3109241984234394
P-value: 0.18988337448195103


In [25]:

import numpy as np

p_control = df_clean[df_clean["group"] == "control"]["converted"].mean()
p_treatment = df_clean[df_clean["group"] == "treatment"]["converted"].mean()

n_control = (df_clean["group"] == "control").sum()
n_treatment = (df_clean["group"] == "treatment").sum()

difference = p_treatment - p_control

se = np.sqrt(
    p_control * (1 - p_control) / n_control
    + p_treatment * (1 - p_treatment) / n_treatment
)

ci_lower = difference - 1.96 * se
ci_upper = difference + 1.96 * se

print("95% CI Lower:", round(ci_lower * 100, 4), "percentage points")
print("95% CI Upper:", round(ci_upper * 100, 4), "percentage points")


95% CI Lower: -0.3938 percentage points
95% CI Upper: 0.0781 percentage points


In [26]:

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import numpy as np

baseline = 0.12
target = 0.13

effect_size = proportion_effectsize(baseline, target)

power_analysis = NormalIndPower()

required_n = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    ratio=1
)

print("Required users per group:", np.ceil(required_n))


Required users per group: 17164.0


In [27]:

actual_control = group_counts["control"]
actual_treatment = group_counts["treatment"]

print("Required per group:", int(np.ceil(required_n)))
print("Actual Control:", actual_control)
print("Actual Treatment:", actual_treatment)

print("Enough users in both groups:",
      min(actual_control, actual_treatment) >= np.ceil(required_n))


Required per group: 17164
Actual Control: 145274
Actual Treatment: 145310
Enough users in both groups: True


In [28]:

from scipy.stats import norm
import numpy as np

p = 0.12
alpha = 0.05
power = 0.80

z_alpha = norm.ppf(1 - alpha / 2)
z_beta = norm.ppf(power)

n_control = group_counts["control"]
n_treatment = group_counts["treatment"]

mde = (z_alpha + z_beta) * np.sqrt(
    p * (1 - p) * (1/n_control + 1/n_treatment)
)

print("MDE:", round(mde * 100, 4), "percentage points")


MDE: 0.3378 percentage points



## Business Recommendation

### Experiment Decision
Do not roll out the new landing page at this time.

### Key Findings
- Control conversion rate: 12.0386%
- Treatment conversion rate: 11.8808%
- Absolute lift: -0.1578 percentage points
- Relative lift: -1.311%
- P-value: 0.1899
- 95% confidence interval for absolute lift:
  [-0.3938, +0.0781] percentage points

### Business Interpretation
The new landing page showed a slightly lower
observed conversion rate than the existing page.

However, the difference was not statistically
significant at the 5% significance level.

The experiment does not provide sufficient
evidence that the new page improves conversion.

### Recommendation
Retain the existing landing page for now.

Review experiment quality and implementation
costs before deciding whether to run another
experiment with an improved page design.

### Limitations
- The dataset is a public sample, not live
  company data.
- Removing mismatched group/page records may
  affect the causal interpretation.
- A non-significant result does not prove
  the two pages perform equally.
